# Evaluation Notebook

This file performs evaluation of the diffusion-based predictor, stores results in a Zarr file, and generates plots.

## Configuration

In [1]:
# Set the output file.
output_file = "./results/toy_problem.zarr"

## Setup

In [2]:
%reload_ext autoreload
%autoreload 2
from onpolicy.scripts.eval.eval_pettingzoo import main
import os
import time
import numpy as np
from multiprocessing import Process
from args import *

MAX_PROCESSES = 200
GPUS = [0, 1, 2, 3, 4, 5, 6, 7]

def get_gpu_least_used():
    gpu_memory = os.popen("nvidia-smi --query-gpu=memory.used --format=csv").read()
    gpu_memory = gpu_memory.split("\n")[1:-1]
    gpu_memory = [int(x.split()[0]) for x in gpu_memory]
    gpu_memory = [gpu_memory[i] if i in GPUS else float('inf') for i in range(len(gpu_memory))]
    least_used_gpu = gpu_memory.index(min(gpu_memory))
    return least_used_gpu

# Set default arguments for evaluation. Feel free to change these!
default_args = {}
default_args["eval_output_file"] = wrap_arg_val(output_file)
default_args["cuda"] = wrap_arg_val(True)
default_args["cuda_idx"] = wrap_arg_val(2)
default_args["cuda_idx_predictor"] = default_args["cuda_idx"]
default_args["eval_episodes"] = wrap_arg_val(100)
default_args["episode_length"] = wrap_arg_val(200)
default_args["seed"] = wrap_arg_val(42)

default_args["observation_mask"] = wrap_arg_val(True)
default_args["random_start_positions"] = wrap_arg_val(False)
default_args["diffusion_autoregression_steps"] = wrap_arg_val(1)
# default_args["prediction_during_training"] = wrap_arg_val(True)

# Evaluation

In [3]:
# Check if the output file already exists. If so, skip this cell.
if os.path.exists(output_file):
    print("Output file already exists. Skipping evaluation.")
    # print("Output file already exists. Overwriting evaluation.")

else:
    process_args = []

    def add_process(args):
        process_args.append(args)
    
    def start_process(args):
        p = Process(target=main, args=(args,))
        p.start()
        return p
    
    # obs_probabilities = np.arange(0.0, 1.0, 0.01).tolist()
    obs_probabilities = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]

    # Add diffuser series.
    for obs_prob in obs_probabilities:
        # We actually just want to compare the predictors - not the policies that use them.
        # Turn off prediction injection, but still run the predictor.
        args = load_args("/data/group/mas/qmas/results/toy_problem_v1/mappo/toy-problem-draft1/wandb/run-20260513_220732-tv5hrfgb/files", default_args)
        args["eval_series"] = "Diffusion Prediction"
        args["experiment_name"] = wrap_arg_val(f"obs_prob_{obs_prob:.2f}")
        args["observation_probability"] = wrap_arg_val(obs_prob)
        args["prediction_during_training"] = wrap_arg_val(False)
        args["eval_prediction"] = wrap_arg_val(True)
        add_process(args)

        # Add KF series.
        args = load_args("/data/group/mas/qmas/results/toy_problem_v1/mappo/toy-problem-draft1/wandb/run-20260513_220732-tv5hrfgb/files", default_args)
        args["eval_series"] = "Kalman Filter Prediction"
        args["observation_probability"] = wrap_arg_val(obs_prob)
        args["experiment_name"] = wrap_arg_val(f"obs_prob_{obs_prob:.2f}")
        args["prediction_history_window"] = wrap_arg_val(1)
        args["prediction_ensemble_size"] = wrap_arg_val(1)
        args["algorithm_class"] = wrap_arg_val("")
        args["policy_class"] = wrap_arg_val("qmas.baseline_kf.policy.QmasPolicy")
        args["prediction_history_include_actions"] = wrap_arg_val(True)
        args["prediction_during_training"] = wrap_arg_val(False)
        args["eval_prediction"] = wrap_arg_val(True)
        add_process(args)


        # No prediction
        args = load_args("/data/group/mas/qmas/results/toy_problem_v1/mappo/toy-problem-draft1/wandb/run-20260513_220732-tv5hrfgb/files", default_args)
        args["eval_series"] = "Observations Only --> Policy"
        args["experiment_name"] = wrap_arg_val(f"obs_prob_{obs_prob:.2f}")
        args["observation_probability"] = wrap_arg_val(obs_prob)
        args["prediction_during_training"] = wrap_arg_val(False)
        add_process(args)

        # Diffusion (injected)
        args = load_args("/data/group/mas/qmas/results/toy_problem_v1/mappo/toy-problem-draft1/wandb/run-20260513_220732-tv5hrfgb/files", default_args)
        args["eval_series"] = "Diffusion Prediction --> Policy"
        args["experiment_name"] = wrap_arg_val(f"obs_prob_{obs_prob:.2f}")
        args["observation_probability"] = wrap_arg_val(obs_prob)
        args["prediction_during_training"] = wrap_arg_val(True)
        add_process(args)

        # KF (injected)
        args = load_args("/data/group/mas/qmas/results/toy_problem_v1/mappo/toy-problem-draft1/wandb/run-20260513_220732-tv5hrfgb/files", default_args)
        args["eval_series"] = "Kalman Filter Prediction --> Policy"
        args["observation_probability"] = wrap_arg_val(obs_prob)
        args["experiment_name"] = wrap_arg_val(f"obs_prob_{obs_prob:.2f}")
        args["prediction_history_window"] = wrap_arg_val(1)
        args["prediction_ensemble_size"] = wrap_arg_val(1)
        args["algorithm_class"] = wrap_arg_val("")
        args["policy_class"] = wrap_arg_val("qmas.baseline_kf.policy.QmasPolicy")
        args["prediction_history_include_actions"] = wrap_arg_val(True)
        args["prediction_during_training"] = wrap_arg_val(True)
        add_process(args)

    # JOIN ALL PROCESSES
    # Up to MAX_PROCESSES processes can run simultaneously.
    running_processes = []
    time_start = time.time()
    failed_processes = []
    
    while len(process_args) > 0 or len(running_processes) > 0:
        # Start new processes if we have capacity.
        while len(running_processes) < MAX_PROCESSES and len(process_args) > 0:
            args = process_args.pop(0)

            # Get least used GPU and set it in args.
            least_used_gpu = get_gpu_least_used()
            args["cuda_idx"] = wrap_arg_val(least_used_gpu)
            args["cuda_idx_predictor"] = wrap_arg_val(least_used_gpu)

            p = start_process(convert_args(args))
            running_processes.append(p)
        
        # Check for finished processes.
        for p in running_processes:
            if not p.is_alive():
                p.join()
                print(f"Process {p.pid} finished with exit code {p.exitcode}. Running: {len(running_processes)-1}, queued: {len(process_args)}, failed: {len(failed_processes)}")
                if p.exitcode != 0:
                    failed_processes.append(p)
        running_processes = [p for p in running_processes if p.is_alive()]
    
    time_end = time.time()
    print(f"Total evaluation time: {time_end - time_start:.2f} seconds")
    
    if failed_processes:
        raise RuntimeError(f"{len(failed_processes)} subprocess(es) failed with non-zero exit codes")
    
    print("All processes finished successfully!")

Output file already exists. Skipping evaluation.


# Analysis

In [4]:
import zarr
output = zarr.open(output_file, mode='r')
output.tree()

Tree(nodes=(Node(disabled=True, name='/', nodes=(Node(disabled=True, name='Diffusion Prediction', nodes=(Node(…

In [5]:
%reload_ext autoreload
%autoreload 2
from plots import *
import numpy as np

x_key = 'observation_probability'

# SERIES_COMPARE_PREDICTION = output.keys()
SERIES_COMPARE_PREDICTION = ["Diffusion Prediction", "Kalman Filter Prediction"]
# SERIES_COMPARE_PERFORMANCE = output.keys()
SERIES_COMPARE_PERFORMANCE = ["Diffusion Prediction --> Policy", "Kalman Filter Prediction --> Policy", "Observations Only --> Policy"]

# Data for plotting
plots = {
    'tracking_error_mean': {
        'title': 'Tracking Error',
        'ylabel': 'Tracking Error (units distance)',
        'xlabel': 'Observation Probability',
        'x': {series: get_x_radii(series, output, x_key) for series in SERIES_COMPARE_PERFORMANCE},
        # Average over all episodes and all timesteps.
        'y': {
            series: [np.mean(output[series][experiment]['tracking_error_mean']) for experiment in output[series]]
                for series in SERIES_COMPARE_PERFORMANCE
        },
        # 'y_std': {
        #     series: [np.std(np.mean(output[series][exp]['prediction_error_mean'], axis=tuple(range(1, output[series][exp]['prediction_error_mean'].ndim)))) for exp in sorted_experiments(series, output, x_key)]
        #         for series in SERIES_COMPARE_PERFORMANCE if not next(iter(output[series].values())).attrs['prediction_disable']
        # }
    },
    'prediction_error_mean': {
        'title': 'Prediction Error',
        'ylabel': 'Prediction Error (units distance)',
        'xlabel': 'Observation Probability',
        'x': {series: get_x_radii(series, output, x_key) for series in SERIES_COMPARE_PREDICTION},
        # Average over all episodes and all timesteps.
        'y': {
            series: [np.mean(output[series][exp]['prediction_error_mean']) for exp in sorted_experiments(series, output, x_key)]
                for series in SERIES_COMPARE_PREDICTION if not next(iter(output[series].values())).attrs['prediction_disable']
        },
        # 'y_std': {
        #     series: [np.std(np.mean(output[series][exp]['prediction_error_mean'], axis=tuple(range(1, output[series][exp]['prediction_error_mean'].ndim)))) for exp in sorted_experiments(series, output, x_key)]
        #         for series in SERIES if not next(iter(output[series].values())).attrs['prediction_disable']
        # }
    },
    'prediction_trajectory_error_mean': {
        'title': 'Prediction Trajectory Error',
        'ylabel': 'Prediction Trajectory Error (units distance)',
        'xlabel': 'Observation Probability',
        'x': {series: get_x_radii(series, output, x_key) for series in SERIES_COMPARE_PREDICTION},
        # Average over all episodes and all timesteps.
        'y': {
            series: [np.mean(output[series][exp]['prediction_trajectory_error_mean']) for exp in sorted_experiments(series, output, x_key)]
                for series in SERIES_COMPARE_PREDICTION if not next(iter(output[series].values())).attrs['prediction_disable']
        }
    },
    'prediction_uncertainty_mean': {
        'title': 'Prediction Uncertainty',
        'ylabel': 'Prediction Uncertainty (units distance)',
        'xlabel': 'Observation Probability',
        'x': {series: get_x_radii(series, output, x_key) for series in SERIES_COMPARE_PREDICTION},
        # Average over all episodes and all timesteps.
        'y': {
            series: [np.mean(output[series][exp]['prediction_uncertainty_mean']) for exp in sorted_experiments(series, output, x_key)]
                for series in SERIES_COMPARE_PREDICTION if not next(iter(output[series].values())).attrs['prediction_disable']
        }
    },
    'prediction_uncertainty_gap_mean': {
        'title': 'Prediction Uncertainty Error',
        'ylabel': 'Prediction Uncertainty Error (units distance)',
        'xlabel': 'Observation Probability',
        'x': {series: get_x_radii(series, output, x_key) for series in SERIES_COMPARE_PREDICTION},
        # Average over all episodes and all timesteps.
        'y': {
            series: [np.mean(output[series][exp]['prediction_uncertainty_gap_mean']) for exp in sorted_experiments(series, output, x_key)]
                for series in SERIES_COMPARE_PREDICTION if not next(iter(output[series].values())).attrs['prediction_disable']
        }
    },
    'prediction_uncertainty_gap_min': {
        'title': 'Prediction Uncertainty Error (Min)',
        'ylabel': 'Prediction Uncertainty Error Min (units distance)',
        'xlabel': 'Observation Probability',
        'x': {series: get_x_radii(series, output, x_key) for series in SERIES_COMPARE_PREDICTION},
        # Average over all episodes and all timesteps.
        'y': {
            series: [np.mean(output[series][exp]['prediction_uncertainty_gap_min']) for exp in sorted_experiments(series, output, x_key)]
                for series in SERIES_COMPARE_PREDICTION if not next(iter(output[series].values())).attrs['prediction_disable']
        }
    },
    'prediction_uncertainty_gap_max': {
        'title': 'Prediction Uncertainty Error (Max)',
        'ylabel': 'Prediction Uncertainty Error Max (units distance)',
        'xlabel': 'Observation Probability',
        'x': {series: get_x_radii(series, output, x_key) for series in SERIES_COMPARE_PREDICTION},
        # Average over all episodes and all timesteps.
        'y': {
            series: [np.mean(output[series][exp]['prediction_uncertainty_gap_max']) for exp in sorted_experiments(series, output, x_key)]
                for series in SERIES_COMPARE_PREDICTION if not next(iter(output[series].values())).attrs['prediction_disable']
        }
    }
}

# Miscelaneous statistics.
timesteps = [np.max([np.max(output[series][experiment]['deltaSteps'].shape[1]) for experiment in output[series]]) for series in SERIES_COMPARE_PREDICTION]

plot(plots, timesteps=timesteps)
